In [1]:
import datetime
import requests
from bs4 import BeautifulSoup
import os
import zipfile

# ───────────────────────────────────────────────────────────────
# PARAMETERS — set your desired date range here
start_date = datetime.date(2020, 12, 28)
end_date   = datetime.date(2021, 2, 6)

# Where to save downloaded files
output_dir = "/data/elugos/2021_coup_DC"
os.makedirs(output_dir, exist_ok=True)
# ───────────────────────────────────────────────────────────────

BASE = "http://data.gdeltproject.org/events/"

# Step 1 — get file list
print("Fetching index...")
resp = requests.get(BASE)
resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")
links = soup.find_all("a")

# helper to parse filenames of form YYYYMMDD.export.CSV.zip
def file_date(fname):
    # e.g. "20250115.export.CSV.zip"
    try:
        dt_str = fname.split(".")[0]
        return datetime.datetime.strptime(dt_str, "%Y%m%d").date()
    except:
        return None

# filter by date range
files_to_download = []
for a in links:
    href = a.get("href")
    if href and href.endswith(".CSV.zip"):
        dt = file_date(href)
        if dt and start_date <= dt <= end_date:
            files_to_download.append(href)

print(f"Found {len(files_to_download)} files in range {start_date} → {end_date}")

# Step 2 — download each file
for fname in sorted(files_to_download):
    url = BASE + fname
    out_path = os.path.join(output_dir, fname)

    if os.path.exists(out_path):
        print(f"✔ Already downloaded: {fname}")
        continue

    print(f"Downloading {fname} ...")
    r = requests.get(url, stream=True)
    r.raise_for_status()

    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"Saved → {out_path}")

print("All downloads done!")

# Step 3 (optional) — unzip
extract = True
if extract:
    for fname in sorted(files_to_download):
        zip_path = os.path.join(output_dir, fname)
        csv_path = zip_path.replace(".zip", "")
        if not os.path.exists(csv_path):
            print(f"Unzipping {fname} ...")
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(output_dir)

    print("Extraction complete!")


Fetching index...
Found 41 files in range 2020-12-28 → 2021-02-06
Saved → /data/elugos/2021_coup_DC/20201228.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20201229.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20201230.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20201231.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210101.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210102.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210103.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210104.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210105.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210106.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210107.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210108.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210109.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210110.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210111.export.CSV.zip
Saved → /data/elugos/2021_coup_DC/20210112.export.CSV.zip
Saved 